In [1]:
# Cell 1: 데이터 초기화 및 GPU 확인
import torch
import gc
import random
import numpy as np

# GPU 메모리 정리
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("GPU 메모리 정리 완료")

# Python 가비지 컬렉션
gc.collect()

# 재현성을 위한 시드 설정
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"랜덤 시드 설정 완료: {RANDOM_SEED}")

# GPU 확인
print('GPU not found.') if not torch.cuda.is_available() else print(f'Found GPU: {torch.cuda.get_device_name(0)}')

GPU 메모리 정리 완료
랜덤 시드 설정 완료: 42
Found GPU: NVIDIA GeForce RTX 3070


In [4]:
# Cell 2: 라이브러리 import
import torch
import numpy as np
from pathlib import Path
from utils.patch_cnn.classifier import initialize_defect_classifier, load_defect_classifier
from utils.patch_cnn.dataset_functions import (
    create_defect_only_patch_dataset, get_label_mapping,
    get_albumentations_transform
)
from utils.patch_cnn.federated_averaging import federated_averaging_defect_classifier
from utils.patch_cnn.visualization import plot_training_curves

In [5]:
# Cell 3: 데이터 경로 및 클라이언트 분배 설정 (분산 평가 방식)
import random

data_dir = 'data_test/'
imagePath0 = f'{data_dir}/0/'
imagePath1 = f'{data_dir}/1/'
npyPath = f'{data_dir}/annotations/'

# 파일 분배 설정
total_files = 78
num_clients = 6
# test_client_id 제거 - 모든 클라이언트가 학습에 참여

all_files = [f'{i:06d}' for i in range(1, total_files + 1)]
random.shuffle(all_files)

# 모든 클라이언트를 학습 클라이언트로 설정
train_clients = [f'client{i}' for i in range(1, num_clients + 1)]
num_train_clients = len(train_clients)
files_per_train_client = len(all_files) // num_train_clients

clientIdentifierDict = {}

start_idx = 0
for i, client_id in enumerate(train_clients):
    if i < num_train_clients - 1:
        end_idx = start_idx + files_per_train_client
    else:
        end_idx = len(all_files)
    
    client_files = all_files[start_idx:end_idx]
    clientIdentifierDict[client_id] = client_files
    
    print(f'{client_id}: {len(client_files)}개 파일')
    start_idx = end_idx

print(f'\n총 학습 클라이언트: {len(train_clients)}개')
print(f'총 학습 데이터: {sum(len(files) for files in clientIdentifierDict.values())}개')
print(f'\n※ 분산 평가 방식: 각 클라이언트가 자신의 데이터를 Train/Val/Test로 분할합니다.')

client1: 13개 파일
client2: 13개 파일
client3: 13개 파일
client4: 13개 파일
client5: 13개 파일
client6: 13개 파일

총 학습 클라이언트: 6개
총 학습 데이터: 78개

※ 분산 평가 방식: 각 클라이언트가 자신의 데이터를 Train/Val/Test로 분할합니다.


In [6]:
# Cell 4: 레이블 매핑 생성
# 전체 레이블 매핑 (모든 레이블 유형 포함)
all_label_mapping, all_num_classes = get_label_mapping(data_dir, clientIdentifierDict)

# 결함 유형만 포함하는 레이블 매핑 생성 (0, 1, -1 제외)
defect_only_label_mapping = {}
defect_classes = []
for original_label, mapped_label in all_label_mapping.items():
    # 0, 1, -1은 정상 부분이므로 제외
    if original_label not in [0, 1, -1]:
        defect_only_label_mapping[original_label] = len(defect_classes)
        defect_classes.append(original_label)

num_defect_classes = len(defect_classes)
print(f"\n결함 유형 클래스 수: {num_defect_classes}")
print(f"결함 유형 매핑: {defect_only_label_mapping}")
print(f"결함 유형 원본 값: {defect_classes}")

발견된 모든 레이블 유형: [np.int8(-1), np.int8(0), np.int8(1), np.int32(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9), np.int8(11), np.int8(14)]
레이블 매핑: {np.int8(-1): 0, np.int8(0): 1, np.int8(1): 2, np.int32(2): 3, np.int8(3): 4, np.int8(4): 5, np.int8(5): 6, np.int8(6): 7, np.int8(7): 8, np.int8(8): 9, np.int8(9): 10, np.int8(11): 11, np.int8(14): 12}
총 클래스 수: 13

결함 유형 클래스 수: 10
결함 유형 매핑: {np.int32(2): 0, np.int8(3): 1, np.int8(4): 2, np.int8(5): 3, np.int8(6): 4, np.int8(7): 5, np.int8(8): 6, np.int8(9): 7, np.int8(11): 8, np.int8(14): 9}
결함 유형 원본 값: [np.int32(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7), np.int8(8), np.int8(9), np.int8(11), np.int8(14)]


In [7]:
# Cell 5: 결함 패치 데이터셋 생성 (결함 유형 분류용)
# 결함 픽셀만 포함하는 패치를 생성하여 결함 유형을 분류하는 모델 학습

PATCH_SIZE = (512, 512)
PATCH_OVERLAP = 0
MIN_DEFECT_RATIO = 0.1  # 패치 내 최소 결함 픽셀 비율
MIN_CONFIDENCE = 0.3  # 최소 신뢰도

# Albumentations 증강 설정
train_transform = get_albumentations_transform(is_training=True)
val_transform = get_albumentations_transform(is_training=False)

print("결함 패치 데이터셋 생성 중...")
print(f"패치 크기: {PATCH_SIZE}")
print(f"최소 결함 비율: {MIN_DEFECT_RATIO}")
print(f"최소 신뢰도: {MIN_CONFIDENCE}")

# 모든 클라이언트의 데이터셋 생성 (증강 적용)
# 분산 평가 방식: 각 클라이언트가 자신의 데이터를 Train/Val/Test로 분할
trainImageDict, trainLabelDict, trainMetadataDict = create_defect_only_patch_dataset(
    clientIdentifierDict,  # 모든 클라이언트 포함
    data_dir,
    patch_size=PATCH_SIZE,
    patch_overlap=PATCH_OVERLAP,
    min_defect_ratio=MIN_DEFECT_RATIO,
    min_confidence=MIN_CONFIDENCE,
    label_mapping=defect_only_label_mapping,
    transform=train_transform
)

# 테스트 데이터셋은 더 이상 별도로 생성하지 않음
# 각 클라이언트가 자신의 데이터를 Train/Val/Test로 분할하여 사용
print("\n※ 분산 평가 방식: 테스트 데이터셋은 각 클라이언트 내에서 분할됩니다.")

d:\iot\IoT_FL_AM\venv311\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
d:\iot\IoT_FL_AM\utils\patch_cnn\dataset_functions.py:131: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(
d:\iot\IoT_FL_AM\utils\patch_cnn\dataset_functions.py:144: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
d:\iot\IoT_FL_AM\utils\patch_cnn\dataset_functions.py:155: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(


결함 패치 데이터셋 생성 중...
패치 크기: (512, 512)
최소 결함 비율: 0.1
최소 신뢰도: 0.3
client1...
원본 파일: 13개
생성된 결함 패치: 36개
증강 적용 후: 288개 (증강 배수: 8)
Image Tensor Shape: torch.Size([288, 2, 512, 512])
Label Tensor Shape: torch.Size([288])
결함 유형 분포: {6: 104, 2: 184}
client2...
원본 파일: 13개
생성된 결함 패치: 76개
증강 적용 후: 608개 (증강 배수: 8)
Image Tensor Shape: torch.Size([608, 2, 512, 512])
Label Tensor Shape: torch.Size([608])
결함 유형 분포: {0: 56, 2: 504, 6: 48}
client3...
원본 파일: 13개
생성된 결함 패치: 60개
증강 적용 후: 480개 (증강 배수: 8)
Image Tensor Shape: torch.Size([480, 2, 512, 512])
Label Tensor Shape: torch.Size([480])
결함 유형 분포: {6: 72, 0: 192, 2: 216}
client4...
원본 파일: 13개
생성된 결함 패치: 45개
증강 적용 후: 360개 (증강 배수: 8)
Image Tensor Shape: torch.Size([360, 2, 512, 512])
Label Tensor Shape: torch.Size([360])
결함 유형 분포: {6: 128, 2: 112, 0: 120}
client5...
원본 파일: 13개
생성된 결함 패치: 84개
증강 적용 후: 672개 (증강 배수: 8)
Image Tensor Shape: torch.Size([672, 2, 512, 512])
Label Tensor Shape: torch.Size([672])
결함 유형 분포: {2: 392, 5: 48, 6: 32, 0: 200}
client6...
원

In [8]:
# Cell 6: 분산 평가 방식 Train/Val/Test로 분할

trainClients = [f'client{i}' for i in range(1, num_clients + 1)]  # 모든 클라이언트 포함

print(f"학습 클라이언트: {trainClients}")
print(f"총 클라이언트 수: {len(trainClients)}")
print("\n※ 분산 평가 방식:")
print("  - 각 클라이언트가 자신의 데이터를 Train(60%)/Val(20%)/Test(20%)로 분할")
print("  - 각 클라이언트가 자신의 테스트 데이터로 평가")
print("  - 서버가 클라이언트별 정확도의 가중 평균 계산")

학습 클라이언트: ['client1', 'client2', 'client3', 'client4', 'client5', 'client6']
총 클라이언트 수: 6

※ 분산 평가 방식:
  - 각 클라이언트가 자신의 데이터를 Train(60%)/Val(20%)/Test(20%)로 분할
  - 각 클라이언트가 자신의 테스트 데이터로 평가
  - 서버가 클라이언트별 정확도의 가중 평균 계산


In [9]:
# Cell 7: 하이퍼파라미터 설정
SERVER_ROUNDS = 20
LOCAL_EPOCHS = 3
LOCAL_BATCH_SIZE = 32
LOCAL_LEARNING_RATE = 1e-4
EARLY_STOPPING_ACCURACY = 95.0
FREEZE_EPOCHS = 3  # 초기 몇 에포크는 backbone 고정
K_FOLDS = 5  # K-Fold 교차 검증

In [10]:
# Cell 8: 모델 초기화
import os
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 저장된 모델 경로 설정
saved_model_dir = Path('saved_models')
model_filename = 'none'  # 원하는 모델 파일명으로 변경 가능
model_path = saved_model_dir / model_filename

# 저장된 모델이 있으면 로드, 없으면 새로 초기화
if model_path.exists():
    print(f"저장된 모델 발견: {model_path}")
    print("모델 로드 중...")
    model = load_defect_classifier(
        model_path=str(model_path),
        num_classes=num_defect_classes,
        device=device
    )
    print(f"저장된 모델 로드 완료! 클래스 수: {num_defect_classes}")
else:
    print("저장된 모델이 없습니다. 새 모델을 초기화합니다.")
    # ResNet-18 기반 결함 유형 분류 모델 초기화
    # ImageNet 사전 학습, 초기에는 backbone 고정
    model = initialize_defect_classifier(
        num_classes=num_defect_classes,
        input_channels=2,
        device=device,
        pretrained=True,
        freeze_backbone=True  # 초기에는 고정
    )
    print(f"결함 유형 분류 모델 초기화 완료! 클래스 수: {num_defect_classes}")

print(f"Device: {device}")

저장된 모델이 없습니다. 새 모델을 초기화합니다.
결함 유형 분류 모델 초기화 완료! 클래스 수: 10
Device: cuda


d:\iot\IoT_FL_AM\venv311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\iot\IoT_FL_AM\venv311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# # Cell 9: K-Fold 교차 검증 실행 (선택사항)
# # 전체 학습 데이터를 하나로 합침
# all_train_images, all_train_labels = unwrap_client_data(
#     trainImageDict, trainLabelDict, trainClients
# )

# # K-Fold 교차 검증 실행
# print("K-Fold 교차 검증 시작...")
# fold_results = k_fold_cross_validation(
#     model, K_FOLDS, all_train_images, all_train_labels, num_classes,
#     SERVER_ROUNDS, LOCAL_EPOCHS, LOCAL_BATCH_SIZE, LOCAL_LEARNING_RATE,
#     device=device,
#     early_stopping_accuracy=EARLY_STOPPING_ACCURACY
# )

# # 결과 시각화
# plot_kfold_results(fold_results)

In [ ]:
# Cell 10: 연합학습 실행 (결함 유형 분류 모델) - 분산 평가 방식
model, serverStateDict, lossDict, testLoss, accuracyDict, testAccuracy = federated_averaging_defect_classifier(
    model,
    SERVER_ROUNDS, LOCAL_EPOCHS, LOCAL_BATCH_SIZE, LOCAL_LEARNING_RATE,
    trainClients, trainImageDict, trainLabelDict,  # testImages, testLabels 제거
    num_defect_classes,
    device=device,
    early_stopping_accuracy=EARLY_STOPPING_ACCURACY,
    freeze_epochs=FREEZE_EPOCHS,
    train_ratio=0.6,  # 60% 학습
    val_ratio=0.2,    # 20% 검증
    test_ratio=0.2    # 20% 테스트
)

In [ ]:
# Cell 11: 학습 곡선 시각화
plot_training_curves(lossDict, accuracyDict, testLoss, testAccuracy, trainClients)

In [ ]:
# Cell 12: 모델 저장
import os
from pathlib import Path
from utils.patch_cnn.classifier import save_patch_cnn_model

os.makedirs('saved_models', exist_ok=True)

lr_str = f"{LOCAL_LEARNING_RATE:.0e}".replace('-', '').replace('+', '').replace('.0', '')
base_filename = f'saved_models/Defect_Classifier_FL_{SERVER_ROUNDS}_{LOCAL_EPOCHS}_{LOCAL_BATCH_SIZE}_{lr_str}.pth'

model_filename = base_filename
counter = 1
while os.path.exists(model_filename):
    base_name, ext = os.path.splitext(base_filename)
    model_filename = f'{base_name}_{counter}{ext}'
    counter += 1

save_patch_cnn_model(model, model_filename, num_classes=num_defect_classes)
print(f'\n최종 테스트 성능:')
print(f'  Loss: {testLoss[-1]:.4f}')
print(f'  Accuracy: {testAccuracy[-1]:.2f}%')
print(f'\n모델 저장 완료: {model_filename}')
print(f'사용된 데이터: {data_dir} ({total_files}개 파일)')

In [3]:
# Cell 13: 테스트 데이터 시각화
from utils.patch_cnn.visualization import visualize_test_results

# 설정 (원하는 값으로 변경 가능)
MODEL_PATH = 'patch_cnn_models/round_3.pth'  # None이면 자동으로 가장 최근 모델 찾기
DATA_DIR = 'data_test/'  # 데이터셋 경로
TOTAL_FILES = 78  # 총 파일 개수

# 시각화 실행
visualize_test_results(
    model_path=MODEL_PATH,
    data_dir=DATA_DIR,
    total_files=TOTAL_FILES,
    patch_size=(512, 512),
    patch_overlap=0,
    output_dir='visualizations',
    subdivide_small_images=True  # 파라미터 이름 변경
)

모델 로드: patch_cnn_models\round_3.pth
결함 유형 클래스 수: 10
결함 유형 매핑: {np.int32(2): 0, np.int8(3): 1, np.int8(4): 2, np.int8(5): 3, np.int32(6): 4, np.int8(7): 5, np.int8(8): 6, np.int8(9): 7, np.int8(11): 8, np.int8(14): 9}
결함 유형 원본 값: [np.int32(2), np.int8(3), np.int8(4), np.int8(5), np.int32(6), np.int8(7), np.int8(8), np.int8(9), np.int8(11), np.int8(14)]


d:\iot\IoT_FL_AM\venv311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\iot\IoT_FL_AM\venv311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



처리 중: 000001 (1/78)
  저장 완료: visualizations/visualization_000001.png

처리 중: 000002 (2/78)
  저장 완료: visualizations/visualization_000002.png

처리 중: 000003 (3/78)
  저장 완료: visualizations/visualization_000003.png

처리 중: 000004 (4/78)
  저장 완료: visualizations/visualization_000004.png

처리 중: 000005 (5/78)
  저장 완료: visualizations/visualization_000005.png

처리 중: 000006 (6/78)
  저장 완료: visualizations/visualization_000006.png

처리 중: 000007 (7/78)
  저장 완료: visualizations/visualization_000007.png

처리 중: 000008 (8/78)
  저장 완료: visualizations/visualization_000008.png

처리 중: 000009 (9/78)
  저장 완료: visualizations/visualization_000009.png

처리 중: 000010 (10/78)
  저장 완료: visualizations/visualization_000010.png

처리 중: 000011 (11/78)
  저장 완료: visualizations/visualization_000011.png

처리 중: 000012 (12/78)
  저장 완료: visualizations/visualization_000012.png

처리 중: 000013 (13/78)
  저장 완료: visualizations/visualization_000013.png

처리 중: 000014 (14/78)
  저장 완료: visualizations/visualization_000014.png

처리 중: 000015 (